# **Install and/or import libraries:**

Install (and import) these libraries on the machine if you already dont have:

**On Colab:**

In [1]:
'''
!pip install mpi4py
!pip install noise
'''

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.2/466.2 kB 10.8 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for mpi4py (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [79 lines of output]
      running bdist_wheel
      running build
      running build_src
      using Cython 3.0.11
      cythonizing 'src/mpi4py/MPI.pyx' -> 'src/mpi4py/MPI.c'
      running build_py
      creating build/lib.linux-x86_64-cpython-310/mpi4py
      copying src/mpi4py/bench.py -> build/lib.linux-x86_64-cpython-310/mpi4py
      copying src/mpi4py/__init__.py -> build/lib.linux-x86_64-cpython-310/mpi4py
      copying src/mpi4py/run.py -> build/lib.linux-x86_64-cpython-310/mpi4py
      copying src/mpi4py/typing.py -> build/lib.linux-x86_64-cpython-310/mpi4py
      copying src/mpi4p

**On Kaggle:**

In [1]:
! apt-get install -y libopenmpi-dev
! pip install wheel mpi4py
! pip install noise

Reading package lists... Done
Building dependency tree       
Reading state information... Done
The following additional packages will be installed:
  autoconf automake autotools-dev cpp-8 file gcc-8 gcc-8-base gfortran
  gfortran-8 gfortran-9 ibverbs-providers libcaf-openmpi-3 libcoarrays-dev
  libcoarrays-openmpi-dev libevent-core-2.1-7 libevent-dev
  libevent-extra-2.1-7 libevent-openssl-2.1-7 libevent-pthreads-2.1-7
  libfabric1 libgcc-8-dev libgfortran-8-dev libgfortran-9-dev libhwloc-dev
  libhwloc-plugins libhwloc15 libibverbs-dev libltdl-dev libmagic-mgc
  libmagic1 libmpx2 libnl-3-dev libnl-route-3-dev libopenmpi3 libpmix2
  libpsm-infinipath1 libpsm2-2 librdmacm1 libsigsegv2 libtool libxnvctrl0 m4
  openmpi-bin openmpi-common
Suggested packages:
  autoconf-archive gnu-standards autoconf-doc gettext gcc-8-locales
  gcc-8-multilib gcc-8-doc gfortran-multilib gfortran-doc gfortran-8-multilib
  gfortran-8-doc gfortran-9-multilib gfortran-9-doc libhwloc-contrib-plugins
  libtool-d

In [3]:
# Old libraries backup:
'''
try:
    from mpi4py import MPI
except ImportError:
    !pip install mpi4py
    from mpi4py import MPI

try:
    import noise
except ImportError:
    !pip install noise
    import noise

import time
import numpy as np
from tqdm import tqdm
from numpy.fft import fftfreq, fft, ifft, irfft2, rfft2

try: # If you want to use pyfftw
    from pyfftw.interfaces.numpy_fft import fft, ifft, irfft2, rfft2
    import pyfftw
    pyfftw.interfaces.cache.enable()
except ImportError:
    pass
'''

# **Simulation:**

# MPI writefile version:

In [3]:
%%writefile /kaggle/working/test_mpi.py

import time
import noise
import numpy as np
from tqdm import tqdm
from mpi4py import MPI
from numpy.fft import fftfreq, fft, ifft, irfft2, rfft2, fftshift, ifftshift, fftn, irfftn

def fftn_mpi(u, fu):
    '''
    Perform forward Fourier transform using MPI
    '''
    Uc_hatT[:] = rfft2(u, axes=(1, 2))
    fu[:] = np.rollaxis(Uc_hatT.reshape(Np, num_processes, Np, N//2+1), 1).reshape(fu.shape)
    comm.Alltoall(MPI.IN_PLACE, [fu, MPI.DOUBLE_COMPLEX])
    fu[:] = fft(fu, axis=0)
    return fu

def ifftn_mpi(fu, u):
    '''
    Perform inverse Fourier transform using MPI
    '''
    Uc_hat[:] = ifft(fu, axis=0)
    comm.Alltoall(MPI.IN_PLACE, [Uc_hat, MPI.DOUBLE_COMPLEX])
    Uc_hatT[:] = np.rollaxis(Uc_hat.reshape((num_processes, Np, Np, N//2+1)), 1).reshape(Uc_hatT.shape)
    u[:] = irfft2(Uc_hatT, axes=(1, 2))
    return u

def ifftn_serial(fu, u):
    '''
    Perform inverse Fourier transform (serial version)
    '''
    Uc_hat[:] = ifft(fu, axis=0)
    #comm.Alltoall(MPI.IN_PLACE, [Uc_hat, MPI.DOUBLE_COMPLEX])
    Uc_hatT[:] = np.rollaxis(Uc_hat.reshape((num_processes, N, Np, N//2+1)), 1).reshape(Uc_hatT.shape)
    u[:] = irfft2(Uc_hatT, axes=(1, 2))
    return u

def Cross(a, b, c):
    '''
    Compute the cross product of two vectors
    '''
    c[0] = fftn_mpi(a[1]*b[2]-a[2]*b[1], c[0])
    c[1] = fftn_mpi(a[2]*b[0]-a[0]*b[2], c[1])
    c[2] = fftn_mpi(a[0]*b[1]-a[1]*b[0], c[2])
    return c

def Curl(a, c):
    '''
    Compute the curl of a vector field
    '''
    c[2] = ifftn_mpi(1j*(K[0]*a[1]-K[1]*a[0]), c[2])
    c[1] = ifftn_mpi(1j*(K[2]*a[0]-K[0]*a[2]), c[1])
    c[0] = ifftn_mpi(1j*(K[1]*a[2]-K[2]*a[1]), c[0])
    return c

def ComputeRHS(dU, rk):
    '''
    Compute the right-hand side of the Navier-Stokes equations
    '''
    if rk > 0:
        for i in range(3):
            U[i] = ifftn_mpi(U_hat[i], U[i])
    curl[:] = Curl(U_hat, curl)
    dU = Cross(U, curl, dU)
    dU *= dealias
    P_hat[:] = np.sum(dU*K_over_K2, 0, out=P_hat)
    dU -= P_hat*K
    dU -= viscosity*K2*U_hat
    return dU

def IC_3D(X, IC_type):
    '''
    This function initializes the velocity field in Fourier space based on the initial condition type
    '''
    if IC_type == 'random_vel':
        # Random velocity initial condition (not a very good IC for 3D turbulence)
        U[0] = np.random.rand(*X[0].shape)
        U[1] = np.random.rand(*X[0].shape)
        U[2] = np.random.rand(*X[0].shape)

        #Resize:
        U[0] /= np.max(U[0])
        U[1] /= np.max(U[1])
        U[2] /= np.max(U[2])

    if IC_type == 'taylor_green':
        # Taylor-Green vortex initial conditions (Check Mortensen (2016) paper)
        U[0] = np.sin(X[0])*np.cos(X[1])*np.cos(X[2])
        U[1] = -np.cos(X[0])*np.sin(X[1])*np.cos(X[2])
        U[2] = 0

    if IC_type == 'taylor_green_noise':
        # Taylor-Green vortex with added noise initial condition
        U[0] = np.sin(X[0])*np.cos(X[1])*np.cos(X[2])
        U[1] = -np.cos(X[0])*np.sin(X[1])*np.cos(X[2])
        U[2] = 0

        #Add white noise:
        epsilon = 0.1
        U[0] += epsilon*np.random.rand(*U[0].shape)
        U[1] += epsilon*np.random.rand(*U[1].shape)
        U[2] += epsilon*np.random.rand(*U[2].shape)

    if IC_type == 'perlin_noise':
        # Perlin noise CURL initial condition

        scale = 1/4 #0.25
        octaves = 5 #2
        persistence = 0.4 #0.5
        lacunarity = 2 #2

        for i in range(X[0].shape[0]):
            for j in range(X[0].shape[1]):
                for k in range(X[0].shape[2]):
                    noise_value = noise.pnoise3(X[0][i, j, k]*scale,
                                                X[1][i, j, k]*scale,
                                                X[2][i, j, k]*scale,
                                                octaves=octaves,
                                                persistence=persistence,
                                                lacunarity=lacunarity)
                    # A scalar perlin noise field is generated, then, the same values are assigned to every velocity component.
                    U[0][i, j, k] = noise_value
                    U[1][i, j, k] = noise_value
                    U[2][i, j, k] = noise_value

    # On spectral space:
    for i in range(3):
        U_hat[i] = fftn_mpi(U[i], U_hat[i])

    return U, U_hat

import seaborn as sns
import matplotlib.pyplot as plt

def plot(P, U, curl, t, T, step, Nstep): #plot(P, u, v, U, curl, t, T, step, Nstep): # THIS IS OUTDATED

    sns.set_style('whitegrid')

    #Extract the slices from the dataset to plot
    pressure_slices = [P[:, :, -1],
                       P[0, :, :],
                       P[:, -1, :]]

    velocity_slices = [np.linalg.norm(U, axis=0, keepdims=True)[0][:, :, -1],
                       np.linalg.norm(U, axis=0, keepdims=True)[0][0, :, :],
                       np.linalg.norm(U, axis=0, keepdims=True)[0][:, -1, :]]

    curl_slices = [np.linalg.norm(curl, axis=0, keepdims=True)[0][:, :, -1],
                   np.linalg.norm(curl, axis=0, keepdims=True)[0][0, :, :],
                   np.linalg.norm(curl, axis=0, keepdims=True)[0][:, -1, :]]

    data_list = [pressure_slices, velocity_slices, curl_slices]
    titles = ['Pressure', 'Velocity', 'Vorticity']

    # Create a meshgrid for the planes
    a_dim = np.arange(0, P.shape[0])
    b_dim = np.arange(0, P.shape[1])
    A, B = np.meshgrid(a_dim, b_dim)

    # Create the figure
    fig = plt.figure(figsize=(18, 6), dpi=90)
    fig.suptitle(r'$\mathbf{Physical\ Time:}$ ' + f'{t:.2f}/{T}  |  ' + r'$\mathbf{Iteration:}$ ' + f'{step}/{Nstep}', fontsize=16)

    for idx, (slices, title) in enumerate(zip(data_list, titles)):
        ax = fig.add_subplot(1, 3, idx + 1, projection='3d')

        ax.plot_surface(A, B, np.full_like(A, P.shape[0] - 1),
                        facecolors=plt.cm.viridis(slices[0] / np.max(slices[0])),
                        rstride=1, cstride=1, shade=False)
        ax.plot_surface(B, np.full_like(A, 0), A,
                        facecolors=plt.cm.viridis(slices[1] / np.max(slices[1])),
                        rstride=1, cstride=1, shade=False)
        ax.plot_surface(np.full_like(A, P.shape[0] - 1), B, A,
                        facecolors=plt.cm.viridis(slices[2] / np.max(slices[2])),
                        rstride=1, cstride=1, shade=False)
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel('X', fontweight="bold")
        ax.set_ylabel('Y', fontweight="bold")
        ax.set_zlabel('Z', fontweight="bold")

    # Save and/or show the figure
    nome_tmp = 'plot_'+str(step)+'.png'
    plot_filenames.append(nome_tmp)
    plt.savefig('/kaggle/working/'+nome_tmp)
    plt.close(fig)
    #plt.show()

#List to save plots:
plot_filenames = []

# SIMULATION PARAMETERS:
viscosity = 1/1600         # Viscosity = 1/Reynolds. Suggestion: Reynolds -> 1600
t_f = 80                   # Final physical time
dt = 0.04                  # Time step. Suggestion: 0.05, 0.15 or 0.01
N = 2**8                   # Grid dimension
IC_type = 'taylor_green'          # Initial conditions. Check initial_conditions.py

# MPI SETUP:
comm = MPI.COMM_WORLD
num_processes = comm.Get_size()
rank = comm.Get_rank()
Np = N//num_processes
print('num_processes', num_processes, 'rank', rank)

n_steps = int(np.ceil(t_f/dt)) #Number of frames in the simulation

# Coordinates and wave numbers:
X = np.mgrid[rank*Np:(rank+1)*Np, :N, :N].astype(float)*2*np.pi/N # 2*pi is the lenght of the physical domain
kx = fftfreq(N, 1./N)
kz = kx[:(N//2+1)].copy()
kz[-1] *= -1
K = np.array(np.meshgrid(kx, kx[rank*Np:(rank+1)*Np], kz, indexing='ij'), dtype=int)
K2 = np.sum(K*K, 0, dtype=int)
K_over_K2 = K.astype(float)/np.where(K2 == 0, 1, K2).astype(float)

# Define dealias:
kmax_dealias = 2./3.*(N//2+1)
dealias = np.array((abs(K[0]) < kmax_dealias)*(abs(K[1]) < kmax_dealias)*
                (abs(K[2]) < kmax_dealias), dtype=bool)

# Preallocate arrays
U = np.empty((3, Np, N, N))
U_hat = np.empty((3, N, Np, N//2+1), dtype=complex)
P = np.empty((Np, N, N))
P_hat = np.empty((N, Np, N//2+1), dtype=complex)
U_hat0 = np.empty((3, N, Np, N//2+1), dtype=complex)
U_hat1 = np.empty((3, N, Np, N//2+1), dtype=complex)
dU = np.empty((3, N, Np, N//2+1), dtype=complex)
Uc_hat = np.empty((N, Np, N//2+1), dtype=complex)
Uc_hatT = np.empty((Np, N, N//2+1), dtype=complex)
curl = np.empty((3, Np, N, N))

# Runge-Kutta coefficients:
a = [1./6., 1./3., 1./3., 1./6.]
b = [0.5, 0.5, 1.]

# --- MAIN LOOP: ---
pbar = tqdm(total=int(n_steps))

# Initial velocity initial field (in physical and spectral space):
U, U_hat = IC_3D(X, IC_type)
'''
# Plot the initial state of the simulation (Obs.: only the velocity was initiated):
P_all = comm.gather(P)
U_all = comm.gather(U)
curl_all = comm.gather(curl)

if rank == 0:

    P_all = np.concatenate(P_all, axis=0)
    U_all = np.concatenate(U_all, axis=1)
    curl_all = np.concatenate(curl_all, axis=1)

    plot(np.ones_like(P_all), U_all, np.ones_like(curl_all), float('NaN'), t_f, float('NaN'), n_steps)
'''
for n in range(n_steps + 1):

    #Initialize U_hat1 and U_hat0 (copies of U_hat used on intermediate steps of Runge-Kutta)
    U_hat1[:] = U_hat0[:] = U_hat

    #Runge-Kutta integration
    for rk in range(4):
        dU = ComputeRHS(dU, rk)                #Compute the right-hand side of N-S equations
        if rk < 3:
            U_hat[:] = U_hat0 + b[rk]*dt*dU    # Update U_hat based on R-K coefficients b
        U_hat1[:] += a[rk]*dt*dU               # Update U_hat1 based on R-K coefficients a

    U_hat[:] = U_hat1[:]                       # Update U-hat with the final results on U_hat1

    # for i in range(3):                       #IS THIS REALLY NEEDED? TO PLOT ONLY? ITS ALREADY CALCULATED ON THE RHS FUNCTION
    #     U[i] = ifftn_mpi(U_hat[i], U[i])     # Transform back to physical space
    
    # Plot/Save the vorticity field at intervals
    if n%3==0 and n!=0: #Suggestion: plot every 150 iterations to animate

        #print('iteração:', n)
        
        # Gather data from other processes
        P_all = ifftn_mpi(P_hat, P) # First, convert the pressure from the spectral to the physical space
        P_all = comm.gather(P)
        P_hat_all = comm.gather(P_hat)
        U_all = comm.gather(U)
        curl_all = comm.gather(curl)
        
        
        #print('delimiter 1')
        #print(P_all)
        #print('delimiter 2')
        #print(P_hat_all)
        #print('delimiter 3')
        #print(U_all)
        #print('delimiter 4')
        #print(curl_all)
        
        # Gather data from all ranks
        #comm.Gather(P_hat, P_all, root=0)  # Gather pressure data
        #comm.Gather(U, U_all, root=0)  # Gather velocity data
        #comm.Gather(curl, curl_all, root=0)  # Gather curl data
        
        if rank == 0:

            #print(np.shape(U_hat)) xxx
            #print(np.shape(P_all[0]), np.shape(P_all[1]))
            #print(np.shape(P_hat_all[0]), np.shape(P_hat_all[1]))
            #print(np.shape(U_all[0]), np.shape(U_all[1]))
            #print(np.shape(curl_all[0]), np.shape(curl_all[1]))
            
            # Concatenate gathered data:
            P_all = np.concatenate(P_all, axis=0)
            P_hat_all = np.concatenate(P_hat_all, axis=1)
            U_all = np.concatenate(U_all, axis=1)
            curl_all = np.concatenate(curl_all, axis=1)
            
            #print('passed') xxx
            #print(np.shape(P_all)) xxx
            #print(np.shape(P_hat_all)) xxx
            #print(np.shape(U_all)) xxx
            #print(np.shape(curl_all)) xxx

            #Inverse fft without MPI to plot the pressure:
            #Uc_hat[:] = ifft(P_hat_all, axis=0)
            #Uc_hatT[:] = Uc_hat
            #P_all[:] = irfft2(Uc_hatT, axes=(1, 2))
            #P_all = fftshift(fft2(ifftshift(P_all))).real
            
            #P_all = ifftn_serial(P_hat_all, P_all)
            #P_all[:] = irfftn(ifftshift(P_hat), s=(N, N, N))
            #print(np.shape(P_all))
            
            plot(P_all, U_all, curl_all, n*dt, t_f, n, n_steps)
            #print('end of plot...')
    
    pbar.update(1)

pbar.close()


Overwriting /kaggle/working/test_mpi.py


# Run on Colab/Kaggle:

**COLAB:**

In [25]:
!mpiexec --use-hwthread-cpus --allow-run-as-root -n 2 python test_mpi.py

**KAGGLE:**

In [27]:
!mpiexec --allow-run-as-root -np 32 python test_mpi.py

num_processes 32 rank 21
  0%|          | 0/2000 [00:00<?, ?it/s]num_processes 32 rank 23
num_processes 32 rank 17
num_processes 32 rank 27
num_processes 32 rank 1
num_processes 32 rank 16
num_processes 32 rank 19
num_processes 32 rank 31
num_processes 32 rank 15
  0%|          | 0/2000 [00:00<?, ?it/s]num_processes 32 rank 3
num_processes 32 rank 13
  0%|          | 0/2000 [00:00<?, ?it/s]num_processes 32 rank 25
num_processes 32 rank 29
num_processes 32 rank 28
num_processes 32 rank 14
num_processes 32 rank 18
num_processes 32 rank 12
  0%|          | 0/2000 [00:00<?, ?it/s]num_processes 32 rank 4
num_processes 32 rank 22
  0%|          | 0/2000 [00:00<?, ?it/s]num_processes 32 rank 8
num_processes 32 rank 30
num_processes 32 rank 26
num_processes 32 rank 2
  0%|          | 0/2000 [00:00<?, ?it/s]num_processes 32 rank 0
num_processes 32 rank 24
  2%|▏         | 40/2000 [00:56<46:22,  1.42s/it] ^C


## **MPI Test (delete):**

In [5]:
%%writefile test_mpi.py

from mpi4py import MPI

# Initialize MPI
comm = MPI.COMM_WORLD  # Get the global communicator
rank = comm.Get_rank()  # Get the rank of the process
size = comm.Get_size()  # Get the total number of processes

# Print "Hello, World" from each process
print(f"Hello, World! from rank {rank} out of {size} processes.")

Overwriting test_mpi.py


In [30]:
%%writefile test_mpi.py

from mpi4py import MPI

comm = MPI.COMM_WORLD  # Initialize the MPI communicator
rank = comm.Get_rank()  # Get the rank (process ID)
size = comm.Get_size()  # Get the number of processes

if size != 2:
    raise ValueError("This example requires exactly 2 processes to run")

if rank == 0:
    # Process 0 sends data to Process 1
    data_to_send = {'message': 'Hello from Process 0', 'number': 123}
    comm.send(data_to_send, dest=1, tag=11)
    print(f"Process {rank} sent data: {data_to_send}")

    # Receive the modified data back from Process 1
    received_data = comm.recv(source=1, tag=22)
    print(f"Process {rank} received data back: {received_data}")

elif rank == 1:
    # Process 1 receives data from Process 0
    received_data = comm.recv(source=0, tag=11)
    print(f"Process {rank} received data: {received_data}")

    # Modify the received data
    received_data['message'] = 'Hello back from Process 1'
    received_data['number'] += 1

    # Send the modified data back to Process 0
    comm.send(received_data, dest=0, tag=22)
    print(f"Process {rank} sent modified data back: {received_data}")


Overwriting test_mpi.py


In [32]:
! mpirun --allow-run-as-root -np 2 python test_mpi.py

Process 1 received data: {'message': 'Hello from Process 0', 'number': 123}
Process 1 sent modified data back: {'message': 'Hello back from Process 1', 'number': 124}
Process 0 sent data: {'message': 'Hello from Process 0', 'number': 123}
Process 0 received data back: {'message': 'Hello back from Process 1', 'number': 124}
